# yolov1_implementation

출처 : https://github.com/aladdinpersson/Machine-Learning-Collection/tree/master/ML/Pytorch/object_detection/YOLO

1. architecture
2. IOU
3. loss

In [ ]:
!pip install torchinfo

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Architecture

모델 부분:

YOLOv1 클래스는 여섯 개의 컨볼루션 블록과 두 개의 전결합층(fully connected)을 정의합니다.
각 컨볼루션 블록은 nn.Conv2d, nn.LeakyReLU(0.1), nn.MaxPool2d 등을 사용하여 구성되어 있습니다.

In [ ]:
# =========================================================================================
# 이 코드는 YOLOv1 논문에 제시된 아키텍처를 PyTorch로 구현한 것입니다.
# 1. architecture_config: 모델의 '설계도' 역할을 하는 리스트
# 2. create_conv_layers: '설계도'를 읽어 실제 컨볼루션 레이어들을 조립하는 함수
# 3. YOLOv1: 조립된 컨볼루션 레이어와 최종 분류/회귀를 위한 FC 레이어를 합쳐 완성된 모델

# -----------------------------------------------------------------------------------
# 1. YOLOv1 컨볼루션 레이어 설계도 (architecture_config)
# -----------------------------------------------------------------------------------
# 이 리스트는 YOLOv1의 컨볼루션 신경망(CNN) 부분을 어떻게 구성할지에 대한 '레시피'입니다.
# 각 요소의 의미는 다음과 같습니다:
#   - 튜플: (필터 크기, 필터 개수, 스트라이드, 패딩) -> Conv2d + LeakyReLU
#   - "M": MaxPool2d 레이어
#   - 리스트: [ Conv1_튜플, Conv2_튜플, 반복횟수 ] -> 반복되는 블록
architecture_config = [
    (7, 64, 2, 3),      # 7x7 Conv, 64 filters, stride 2, padding 3
    "M",                  # MaxPool, 2x2, stride 2
    (3, 192, 1, 1),
    "M",
    (1, 128, 1, 0),
    (3, 256, 1, 1),
    (1, 256, 1, 0),
    (3, 512, 1, 1),
    "M",
    # [(Conv1), (Conv2), 반복횟수] -> 이 블록을 4번 반복
    [(1, 256, 1, 0), (3, 512, 1, 1), 4],
    (1, 512, 1, 0),
    (3, 1024, 1, 1),
    "M",
    # [(Conv1), (Conv2), 반복횟수] -> 이 블록을 2번 반복
    [(1, 512, 1, 0), (3, 1024, 1, 1), 2],
    (3, 1024, 1, 1),
    (3, 1024, 2, 1), # stride=2 인 컨볼루션
    (3, 1024, 1, 1),
    (3, 1024, 1, 1)
]

In [ ]:
# -----------------------------------------------------------------------------------
# 2. 컨볼루션 레이어 조립 함수 (create_conv_layers)
# -----------------------------------------------------------------------------------
# 위 '설계도(config)'를 입력받아, 실제 PyTorch 레이어들로 구성된 모델을 조립하는 함수입니다.
def create_conv_layers(config, in_channels):
    # 생성될 PyTorch 레이어들을 순서대로 담을 빈 리스트입니다.
    layers = []

    # 설계도(config)의 각 항목을 하나씩 순회하며 레이어를 조립합니다.
    for module in config:
        # 항목이 튜플 형태인 경우: 단일 컨볼루션 블록 (Conv2d + LeakyReLU)
        if type(module) == tuple:
            kernel_size, filters, stride, padding = module
            # nn.Conv2d 레이어를 생성하여 리스트에 추가합니다.
            # in_channels는 이전 레이어의 출력 채널 수(filters)를 받아 동적으로 설정됩니다.
            layers.append(nn.Conv2d(in_channels, filters, kernel_size, stride, padding, bias=False)) # YOLO는 BN을 안쓰므로 bias=False는 선택사항
            # YOLO 논문에서는 활성화 함수로 LeakyReLU(alpha=0.1)를 사용합니다.
            layers.append(nn.LeakyReLU(0.1))
            # 다음 레이어의 입력 채널(in_channels)은 현재 레이어의 출력 채널 수(filters)가 됩니다.
            in_channels = filters

        # 항목이 "M"인 경우: Max Pooling 레이어
        elif module == "M":
            # 2x2 크기, stride 2로 다운샘플링하는 MaxPool 레이어를 추가합니다.
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))

        # 항목이 리스트 형태인 경우: 컨볼루션 블록 반복
        elif type(module) == list:
            conv1, conv2, num_repeats = module
            # 지정된 반복 횟수(num_repeats)만큼 루프를 돕니다.
            for _ in range(num_repeats):
                # 첫 번째 컨볼루션 블록을 추가합니다.
                k, f, s, p = conv1
                layers.append(nn.Conv2d(in_channels, f, k, s, p))
                layers.append(nn.LeakyReLU(0.1))
                in_channels = f

                # 두 번째 컨볼루션 블록을 추가합니다.
                k, f, s, p = conv2
                layers.append(nn.Conv2d(in_channels, f, k, s, p))
                layers.append(nn.LeakyReLU(0.1))
                in_channels = f

    # 완성된 레이어 리스트를 nn.Sequential에 넣어 하나의 모델로 만듭니다.
    # '*'는 리스트의 모든 요소를 각각의 인자로 풀어주는 역할을 합니다.
    return nn.Sequential(*layers)

# -----------------------------------------------------------------------------------
# 3. YOLOv1 모델 클래스
# -----------------------------------------------------------------------------------
# 위에서 만든 컨볼루션 부분과 완전 연결(FC) 부분을 합쳐 YOLOv1 모델을 완성합니다.
class YOLOv1(nn.Module):
    # 모델 생성 시 필요한 설정값들을 받습니다.
    # in_channels: 입력 이미지 채널 수 (보통 RGB=3)
    # num_classes: 예측할 클래스 개수 (C)
    # split_size: 이미지를 나눌 그리드 크기 (S)
    # num_boxes: 각 그리드 셀이 예측할 바운딩 박스 개수 (B)
    def __init__(self, in_channels=3, num_classes=20, split_size=7, num_boxes=2):
        super(YOLOv1, self).__init__()
        # create_conv_layers 함수를 호출하여 설계도에 따라 컨볼루션 '몸통' 부분을 만듭니다.
        self.conv_layers = create_conv_layers(architecture_config, in_channels)

        # 컨볼루션 레이어를 통과한 후의 최종 예측을 담당하는 완전 연결(FC) '머리' 부분을 만듭니다.
        self.fc_layers = nn.Sequential(
            # nn.Flatten(): 컨볼루션을 통과한 3D 특징 맵(예: [1024, 7, 7])을 1D 벡터로 쭉 펼칩니다.
            nn.Flatten(),
            # nn.Linear(...): 펼쳐진 벡터를 입력받아 4096개의 노드로 연결하는 첫 번째 FC 레이어입니다.
            # 입력 크기: 1024(마지막 Conv 채널) * 7(그리드 H) * 7(그리드 W)
            nn.Linear(1024 * 7 * 7, 4096),
            nn.LeakyReLU(0.1),
            # nn.Dropout(0.5): 과적합을 방지하기 위해 학습 시 50%의 노드를 랜덤하게 비활성화합니다.
            nn.Dropout(0.5),
            # nn.Linear(...): 최종 출력을 만드는 마지막 FC 레이어입니다.
            # 출력 크기 공식: S * S * (C + B * 5)
            #   - S*S: 전체 그리드 셀의 개수 (7*7=49)
            #   - C: 각 셀이 예측할 클래스 확률 (20개)
            #   - B*5: 각 셀이 예측할 B개의 바운딩 박스 정보. 각 박스는 (x, y, w, h, confidence) 5개의 값을 가집니다.
            nn.Linear(4096, split_size * split_size * (num_classes + num_boxes * 5))
        )
        # 나중에 출력 텐서를 해석하기 위해 S, B, C 값을 저장해 둡니다.
        self.split_size = split_size
        self.num_boxes = num_boxes
        self.num_classes = num_classes

    # 데이터가 모델을 통과하는 순서를 정의합니다.
    def forward(self, x):
        # 1. 입력 이미지(x)를 컨볼루션 '몸통'에 통과시켜 특징 맵을 추출합니다.
        x = self.conv_layers(x)
        # 2. 추출된 특징 맵을 FC '머리'에 통과시켜 최종 예측 벡터를 얻습니다.
        x = self.fc_layers(x)
        # 3. 최종 예측 벡터를 반환합니다. (이후 이 벡터를 S x S x (C+B*5) 형태로 재구성하여 사용)
        return x

## IOU

주어진 두 박스의 중심 좌표와 크기를 바탕으로 좌측상단, 우측하단 좌표를 계산하고, 교집합 영역을 통해 IoU를 계산합니다.


In [ ]:
# =========================================================================================
# IoU (Intersection over Union) 계산 함수 상세 해설
# =========================================================================================
# IoU는 객체 탐지(Object Detection)에서 두 개의 바운딩 박스가 얼마나 겹치는지를 측정하는 핵심적인 지표입니다.
# 예측된 박스와 실제 정답 박스의 유사도를 평가하거나, 중복된 예측을 제거(NMS)하는 데 사용됩니다.
# 값은 0 (전혀 겹치지 않음)과 1 (완벽히 일치) 사이입니다.

import torch

def iou(boxes1, boxes2, eps=1e-6):
    """
    YOLOv1에서 사용하는 [중심 x, 중심 y, 너비, 높이] 형식의 바운딩 박스 두 세트에 대한 IoU를 계산합니다.

    Args:
        boxes1 (torch.Tensor): 첫 번째 바운딩 박스 텐서.
        boxes2 (torch.Tensor): 두 번째 바운딩 박스 텐서.
        eps (float): 분모가 0이 되는 것을 방지하기 위한 아주 작은 값.

    Returns:
        torch.Tensor: 계산된 IoU 값 텐서.
    """

    # -----------------------------------------------------------------------------------
    # 1. 좌표 형식 변환: [중심 x, 중심 y, 너비, 높이] -> [x1, y1, x2, y2]
    # -----------------------------------------------------------------------------------
    # 교집합(intersection) 넓이를 계산하기 쉽도록, 중심점 기반 좌표를
    # 좌측 상단(x1, y1)과 우측 하단(x2, y2)의 꼭짓점 기반 좌표로 변환합니다.
    # boxes1[..., 0:1] 처럼 슬라이싱하는 이유:
    #   - '...' (Ellipsis)는 "이전의 모든 차원을 그대로 유지하라"는 의미입니다. (예: 배치, 그리드 차원 등)
    #   - ':1' 처럼 슬라이싱하면 차원이 유지되어 [B, S, S, 1] 형태가 됩니다.
    #     이렇게 하면 나중에 텐서 간의 연산(브로드캐스팅)이 용이해집니다.

    # 박스 1의 좌측 상단 x1 좌표 = 중심 x - (너비 / 2)
    box1_x1 = boxes1[..., 0:1] - boxes1[..., 2:3] / 2
    # 박스 1의 좌측 상단 y1 좌표 = 중심 y - (높이 / 2)
    box1_y1 = boxes1[..., 1:2] - boxes1[..., 3:4] / 2
    # 박스 1의 우측 하단 x2 좌표 = 중심 x + (너비 / 2)
    box1_x2 = boxes1[..., 0:1] + boxes1[..., 2:3] / 2
    # 박스 1의 우측 하단 y2 좌표 = 중심 y + (높이 / 2)
    box1_y2 = boxes1[..., 1:2] + boxes1[..., 3:4] / 2

    # 박스 2에 대해서도 동일한 변환을 수행합니다.
    box2_x1 = boxes2[..., 0:1] - boxes2[..., 2:3] / 2
    box2_y1 = boxes2[..., 1:2] - boxes2[..., 3:4] / 2
    box2_x2 = boxes2[..., 0:1] + boxes2[..., 2:3] / 2
    box2_y2 = boxes2[..., 1:2] + boxes2[..., 3:4] / 2

    # -----------------------------------------------------------------------------------
    # 2. 교집합(Intersection) 영역의 꼭짓점 좌표 계산
    # -----------------------------------------------------------------------------------
    # 두 박스가 겹치는 영역(교집합)의 좌측 상단(x1, y1)과 우측 하단(x2, y2) 좌표를 찾습니다.

    # 교집합의 좌측 상단 x1 좌표는 두 박스의 x1 좌표 중 '더 큰(오른쪽)' 값입니다.
    x1 = torch.max(box1_x1, box2_x1)
    # 교집합의 좌측 상단 y1 좌표는 두 박스의 y1 좌표 중 '더 큰(아래쪽)' 값입니다.
    y1 = torch.max(box1_y1, box2_y1)
    # 교집합의 우측 하단 x2 좌표는 두 박스의 x2 좌표 중 '더 작은(왼쪽)' 값입니다.
    x2 = torch.min(box1_x2, box2_x2)
    # 교집합의 우측 하단 y2 좌표는 두 박스의 y2 좌표 중 '더 작은(위쪽)' 값입니다.
    y2 = torch.min(box1_y2, box2_y2)

    # -----------------------------------------------------------------------------------
    # 3. 교집합(Intersection) 및 각 박스의 넓이 계산
    # -----------------------------------------------------------------------------------
    # torch.clamp(..., min=0): 텐서의 모든 원소 값이 최소 0 이상이 되도록 만듭니다.
    #   - 만약 두 박스가 겹치지 않는다면, (x2 - x1) 또는 (y2 - y1)이 음수가 될 수 있습니다.
    #   - 이 경우, clamp 함수는 음수 값을 0으로 만들어 교집합의 넓이가 0이 되도록 정확하게 계산해줍니다.
    inter = torch.clamp(x2 - x1, min=0) * torch.clamp(y2 - y1, min=0)

    # 각 박스의 전체 넓이를 계산합니다. (너비 * 높이)
    box1_area = torch.abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = torch.abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    # -----------------------------------------------------------------------------------
    # 4. 최종 IoU 값 계산
    # -----------------------------------------------------------------------------------
    # IoU = 교집합 넓이 / 합집합 넓이
    # 합집합 넓이 공식: Union(A, B) = Area(A) + Area(B) - Intersection(A, B)
    # 분모에 아주 작은 값(eps)을 더해, 박스들의 넓이가 0일 때 0으로 나누는 에러를 방지합니다.
    iou_val = inter / (box1_area + box2_area - inter + eps)

    return iou_val

### NMS

- confidence가 낮은 박스 제거 후, 가장 높은 confidence 박스를 선택하고 나머지 박스들과 IoU를 계산하여 제거합니다.
- 박스 간의 IoU 계산 시, 텐서 변환 및 .item() 호출로 스칼라 값을 사용합니다.

In [ ]:
# =========================================================================================
# NMS (Non-Maximum Suppression, 비최대 억제) 함수 상세 해설
# =========================================================================================
# NMS는 객체 탐지 모델의 후처리 과정에서 매우 중요한 알고리즘입니다.
# 모델은 종종 하나의 객체에 대해 여러 개의 바운딩 박스를 중복해서 예측하는데,
# NMS는 이 중 가장 신뢰도(confidence)가 높은 박스 하나만 남기고,
# 그 박스와 많이 겹치는 다른 박스들을 '억제'(제거)하는 역할을 합니다.
# 비유: "한 객체에 대한 여러 후보들 중 가장 확실한 대표 후보 하나만 뽑는 과정"

import torch
# 이전 코드에서 정의한 iou 계산 함수가 필요합니다.
from previous_code import iou

def nms(bboxes, iou_threshold, conf_threshold):
    """
    Args:
        bboxes (list): 예측된 바운딩 박스들의 리스트.
                       각 요소는 [클래스_인덱스, 신뢰도, x1, y1, x2, y2] 형태입니다.
        iou_threshold (float): IoU 임계치. 이 값 이상으로 겹치는 박스는 동일 객체로 간주되어 제거될 수 있습니다.
        conf_threshold (float): 신뢰도 임계치. 이 값보다 낮은 신뢰도를 가진 박스는 시작부터 제거됩니다.
    Returns:
        list: NMS를 거쳐 최종적으로 살아남은 바운딩 박스들의 리스트.
    """

    # -----------------------------------------------------------------------------------
    # 1단계: 신뢰도 임계치(Confidence Threshold) 필터링
    # -----------------------------------------------------------------------------------
    # 제공된 모든 박스들 중에서, 신뢰도(box[1])가 conf_threshold보다 낮은 것들은
    # 아예 고려할 가치가 없다고 판단하고 미리 제거합니다.
    bboxes = [box for box in bboxes if box[1] > conf_threshold]

    # -----------------------------------------------------------------------------------
    # 2단계: 신뢰도 기준 내림차순 정렬
    # -----------------------------------------------------------------------------------
    # NMS 알고리즘은 가장 신뢰도가 높은 박스를 기준으로 나머지를 제거하는 방식입니다.
    # 따라서, 필터링된 박스들을 신뢰도(lambda x: x[1]) 기준으로 내림차순(reverse=True) 정렬합니다.
    # 이제 리스트의 맨 앞(인덱스 0)에는 항상 가장 신뢰도가 높은 박스가 위치하게 됩니다.
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)

    # 최종적으로 선택된 박스들을 담을 빈 리스트를 생성합니다.
    bboxes_nms = []

    # -----------------------------------------------------------------------------------
    # 3단계: 메인 루프 - 가장 확실한 박스를 뽑고, 겹치는 박스들을 제거
    # -----------------------------------------------------------------------------------
    # 처리할 박스('bboxes')가 남아있는 동안 계속해서 루프를 실행합니다.
    while bboxes:
        # 3-1. 가장 신뢰도가 높은 박스를 선택하고, 원본 리스트에서 제거
        # bboxes.pop(0): 정렬된 리스트의 맨 앞에 있는 (가장 신뢰도가 높은) 박스를 꺼냅니다.
        # 이 박스는 이번 라운드의 '챔피언'이며, 최종 결과에 포함될 후보입니다.
        chosen_box = bboxes.pop(0)

        # 3-2. 나머지 박스들을 '챔피언 박스'와 비교하여 살아남을 박스들만 남기기
        # 이 리스트 컴프리헨션이 NMS의 핵심입니다.
        # 남아있는 'bboxes' 리스트를 순회하며, 아래의 '생존 조건'을 만족하는 박스만으로
        # 'bboxes' 리스트를 새로 교체합니다.
        bboxes = [
            box for box in bboxes
            # --- 박스가 살아남기 위한 조건 (아래 둘 중 하나라도 참이면 생존) ---
            # 조건 1: 클래스가 다른가?
            #   - 현재 박스(box)의 클래스(box[0])가 챔피언 박스(chosen_box)의 클래스와 다르면,
            #     서로 다른 객체이므로 겹치더라도 절대 제거하면 안 됩니다. (예: '개'와 '고양이' 박스)
            if box[0] != chosen_box[0]
            # 또는 (OR)
            # 조건 2: 클래스는 같지만, 거의 겹치지 않는가?
            #   - 두 박스의 IoU를 계산했을 때, 그 값이 iou_threshold보다 작으면,
            #     같은 클래스이긴 하지만 서로 다른 객체일 가능성이 높다고 판단합니다. (예: 두 마리의 다른 고양이)
            #     따라서 이 박스도 제거하지 않고 남겨둡니다.
            or iou(
                torch.tensor(chosen_box[2:]).unsqueeze(0),
                torch.tensor(box[2:]).unsqueeze(0)
            ).item() < iou_threshold
            # ---------------------------------------------------------------------
            # 결론: 위 두 조건을 모두 만족하지 못하는 박스, 즉
            # '챔피언 박스와 클래스가 같으면서, IoU가 임계치 이상으로 많이 겹치는 박스'는
            # 중복된 예측으로 간주되어 이 과정에서 자동으로 '제거'(억제)됩니다.
        ]

        # 3-3. 선택된 '챔피언 박스'를 최종 결과 리스트에 추가
        # 이번 라운드에서 살아남은 챔피언 박스를 최종 리스트('bboxes_nms')에 저장합니다.
        bboxes_nms.append(chosen_box)

    # 루프가 끝나면 (더 이상 처리할 박스가 없으면), 중복이 제거된 최종 박스 리스트를 반환합니다.
    return bboxes_nms

## Loss

- 예측값을 [batch, 7, 7, 30] 형태로 재구성한 후, 각 grid cell마다 두 박스의 IoU를 계산하고, best box 선택, 그리고 좌표, confidence, 클래스 손실을 각각 계산하여 최종 loss를 반환합니다.


In [ ]:
# =========================================================================================
# YOLOv1 손실 함수(Loss Function) 상세 해설
# =========================================================================================
# YOLOv1의 손실 함수는 모델의 예측 결과와 실제 정답을 비교하여,
# 모델이 얼마나 '틀렸는지'를 하나의 숫자로 정량화하는 복잡한 수식입니다.
# 이 손실 값(오차)을 최소화하는 방향으로 모델의 가중치가 업데이트(학습)됩니다.
# 이 함수는 크게 3가지 종류의 오차를 모두 계산하여 합산합니다:
#   1. Localization Loss: 바운딩 박스의 위치와 크기가 얼마나 정확한가?
#   2. Confidence Loss: 객체의 존재 유무를 얼마나 잘 예측했는가?
#   3. Classification Loss: 객체의 종류를 얼마나 잘 맞췄는가?

import torch
# 이전 코드에서 정의한 iou 계산 함수가 필요합니다.
from previous_code import iou

def yolo_loss(predictions, target, S=7, B=2, C=20, lambda_coord=5, lambda_noobj=0.5):
    """
    Args:
        predictions (torch.Tensor): 모델의 최종 출력 텐서. 모양: [배치크기, S*S*(C+B*5)]
        target (torch.Tensor): 실제 정답 레이블 텐서. 모양: [배치크기, S, S, C+B*5]
        S, B, C: 그리드 크기, 셀당 박스 수, 클래스 수
        lambda_coord: 좌표 손실에 대한 가중치. 박스 위치의 중요도를 높여줍니다.
        lambda_noobj: 객체가 없는 셀의 신뢰도 손실에 대한 가중치. 대부분의 셀은 배경이므로 중요도를 낮춰줍니다.
    """

    # -----------------------------------------------------------------------------------
    # 0. 예측(predictions) 텐서 재구성
    # -----------------------------------------------------------------------------------
    # 모델 출력은 [B, 1470] 형태의 1차원 벡터이므로, 다루기 쉽도록
    # [B, S, S, C+B*5] 즉, [B, 7, 7, 30] 형태의 그리드 구조로 변환합니다.
    predictions = predictions.view(-1, S, S, C + B * 5)

    # -----------------------------------------------------------------------------------
    # 1. '책임(Responsible)'있는 예측 박스 결정
    # -----------------------------------------------------------------------------------
    # 각 그리드 셀은 B개(2개)의 박스를 예측합니다. 하지만 정답 박스는 1개입니다.
    # 따라서, 2개의 예측 박스 중 정답 박스와 더 많이 겹치는(IoU가 높은) 박스 하나를
    # 이번 학습에서 '책임'을 지도록 선택합니다.

    # 예측 박스 1과 정답 박스의 IoU 계산
    # predictions[..., 21:25] -> 모든 배치, 모든 셀에 대한 첫 번째 박스 [x,y,w,h]
    # target[..., 21:25]      -> 모든 배치, 모든 셀에 대한 정답 박스 [x,y,w,h]
    iou_b1 = iou(predictions[..., 21:25], target[..., 21:25])
    # 예측 박스 2와 정답 박스의 IoU 계산
    # predictions[..., 26:30] -> 모든 배치, 모든 셀에 대한 두 번째 박스 [x,y,w,h]
    iou_b2 = iou(predictions[..., 26:30], target[..., 21:25])

    # 두 IoU 텐서를 하나로 합칩니다.
    ious = torch.cat([iou_b1.unsqueeze(0), iou_b2.unsqueeze(0)], dim=0)

    # torch.max(ious, dim=0): 두 IoU 값 중 더 큰 값(iou_maxes)과
    #                         그 값의 인덱스(bestbox, 0 또는 1)를 찾습니다.
    # bestbox는 어떤 예측 박스가 '책임'을 져야 하는지를 나타내는 마스크가 됩니다.
    iou_maxes, bestbox = torch.max(ious, dim=0)

    # 나중에 마스크로 사용하기 위해 차원을 맞춰줍니다.
    bestbox = bestbox.float().unsqueeze(-1)

    # target[..., 20]은 해당 셀에 객체가 존재하는지 여부(1 또는 0)를 나타냅니다.
    # exists_box는 객체가 '있는' 셀에 대해서만 손실을 계산하기 위한 핵심 마스크입니다.
    exists_box = target[..., 20].unsqueeze(-1)


    # -----------------------------------------------------------------------------------
    # 2. 좌표 손실 (Localization Loss)
    # -----------------------------------------------------------------------------------
    # 오직 '객체가 있는 셀'의 '책임 있는 박스'에 대해서만 위치 및 크기 오차를 계산합니다.

    # (bestbox * pred_box2 + (1-bestbox) * pred_box1) 트릭:
    #   - bestbox가 1이면, (1-bestbox)는 0이 되어 pred_box2만 선택됩니다.
    #   - bestbox가 0이면, (1-bestbox)는 1이 되어 pred_box1만 선택됩니다.
    # exists_box를 곱하여, 객체가 없는 셀의 예측은 0으로 만들어 손실 계산에서 제외합니다.
    box_pred = exists_box * (bestbox * predictions[..., 26:30] + (1 - bestbox) * predictions[..., 21:25])
    box_target = exists_box * target[..., 21:25]

    # 논문에서는 박스의 너비(w)와 높이(h)에 직접 오차를 계산하는 대신, 제곱근(sqrt)을 씌워 계산합니다.
    # 이는 작은 박스의 오차에 더 큰 패널티를 주기 위함입니다.
    # sqrt 계산 시 음수가 들어가는 것을 방지하기 위해 clamp(min=1e-6)로 아주 작은 양수 값을 보장합니다.
    box_pred[..., 2:4] = torch.sign(box_pred[..., 2:4]) * torch.sqrt(torch.abs(box_pred[..., 2:4] + 1e-6))
    box_target[..., 2:4] = torch.sqrt(box_target[..., 2:4])

    # (예측 좌표 - 정답 좌표)의 제곱합을 구하여 최종 좌표 손실을 계산합니다.
    box_loss = torch.sum((box_pred - box_target) ** 2)

    # -----------------------------------------------------------------------------------
    # 3. 신뢰도 손실 (Confidence Loss)
    # -----------------------------------------------------------------------------------

    # --- 3-1. 객체가 '있는' 셀의 신뢰도 손실 (Object Loss) ---
    # 책임 있는 박스의 신뢰도 예측값(pred_conf)을 선택합니다.
    pred_conf = bestbox * predictions[..., 25:26] + (1 - bestbox) * predictions[..., 20:21]
    # 객체가 있을 때, 이상적인 신뢰도 값은 '예측 박스와 정답 박스의 IoU' 입니다.
    confidence_target = exists_box * iou_maxes.unsqueeze(-1)
    # (예측 신뢰도 - 실제 IoU)의 제곱합을 구합니다.
    object_loss = torch.sum((exists_box * (pred_conf - confidence_target)) ** 2)

    # --- 3-2. 객체가 '없는' 셀의 신뢰도 손실 (No-Object Loss) ---
    # (1 - exists_box) 마스크를 사용하여 객체가 '없는' 셀만 선택합니다.
    # 객체가 없을 때 이상적인 신뢰도 값은 '0'입니다.
    # 따라서, 두 예측 박스의 신뢰도 값이 0에서 멀어질수록 패널티를 받습니다.
    noobj_loss = torch.sum(((1 - exists_box) * predictions[..., 20:21]) ** 2) # 박스 1
    noobj_loss += torch.sum(((1 - exists_box) * predictions[..., 25:26]) ** 2) # 박스 2

    # -----------------------------------------------------------------------------------
    # 4. 분류 손실 (Classification Loss)
    # -----------------------------------------------------------------------------------
    # 오직 '객체가 있는 셀'에 대해서만 클래스 예측이 얼마나 틀렸는지 계산합니다.
    # (예측 클래스 확률 - 정답 클래스 확률)의 제곱합을 구합니다.
    class_loss = torch.sum((exists_box * (predictions[..., :20] - target[..., :20])) ** 2)

    # -----------------------------------------------------------------------------------
    # 5. 최종 손실 종합
    # -----------------------------------------------------------------------------------
    # 위에서 계산한 모든 손실들을 가중치를 적용하여 합산합니다.
    total_loss = (
        lambda_coord * box_loss          # 좌표 손실 (가중치 5로 중요도 높임)
        + object_loss                    # 객체 존재 시 신뢰도 손실
        + lambda_noobj * noobj_loss      # 객체 부재 시 신뢰도 손실 (가중치 0.5로 중요도 낮춤)
        + class_loss                     # 분류 손실
    )

    return total_loss

# "손실(loss)이라는 건 목표에 따라 임의로 만들 수도 있다"
#▶ 맞습니다. 이 YOLOv1 손실 함수는 '정확한 위치에, 정확한 크기로, 객체가 있을 때만 있다고 확신하고,
#▶ 그 객체가 무엇인지 정확히 맞추는' 복합적인 목표를 달성하기 위해 위와 같이 정교하게 설계된 것입니다.

In [ ]:
# YOLOv1 모델의 인스턴스를 생성합니다.
# 이 모델은 3채널(RGB) 이미지를 입력받아, 7x7 그리드에 대해 각 셀당 2개의 박스와 20개의 클래스를 예측합니다.
model = YOLOv1(in_channels=3, num_classes=20, split_size=7, num_boxes=2)

# torchinfo.summary를 사용하여 모델의 구조와 파라미터 정보를 출력합니다.
# 입력 크기를 YOLOv1 논문 기준인 448x448로 설정합니다.
summary(model, input_size=(1, 3, 448, 448))

# ==========================================================================================
# Layer (type:depth-idx)                   Output Shape              Param #
# ==========================================================================================
# Layer: 계층의 타입, 깊이(depth)와 순서(idx)를 나타냅니다.
# Output Shape: 해당 계층 통과 후 데이터의 형태. [배치크기, 채널, 높이, 너비] 순서입니다.
# Param #: 해당 계층의 학습 가능한 파라미터(가중치, 편향) 총 개수입니다.

YOLOv1                                   [1, 1470]                 --
#▶ 최종 출력은 1470개의 값을 가진 1차원 벡터입니다.
#▶ 계산: S*S*(C + B*5) = 7*7*(20 + 2*5) = 49 * 30 = 1470

# ------------------------------------------------------------------------------------------
# [1단계] 컨볼루션 레이어 (Backbone)
# ------------------------------------------------------------------------------------------
├─Sequential: 1-1                        [1, 1024, 7, 7]           --
#▶ 이 Sequential 블록은 YOLOv1의 전체 컨볼루션 '몸통' 부분에 해당합니다.
#▶ 입력 이미지 [1, 3, 448, 448]가 최종적으로 [1, 1024, 7, 7] 특징 맵으로 변환됩니다.
│    └─Conv2d: 2-1                       [1, 64, 224, 224]         9,472
#▶ 첫 Conv(7x7, S2). 크기: 448->224. 파라미터: (7*7*3+1)*64 = 9,472
│    └─LeakyReLU: 2-2                    [1, 64, 224, 224]         --
│    └─MaxPool2d: 2-3                    [1, 64, 112, 112]         --
#▶ 크기: 224->112.
│    └─Conv2d: 2-4                       [1, 192, 112, 112]        110,784
│    └─LeakyReLU: 2-5                    [1, 192, 112, 112]        --
│    └─MaxPool2d: 2-6                    [1, 192, 56, 56]          --
#▶ 크기: 112->56.
│    └─Conv2d: 2-7                       [1, 128, 56, 56]          24,704
│    └─LeakyReLU: 2-8                    [1, 128, 56, 56]          --
│    └─Conv2d: 2-9                       [1, 256, 56, 56]          295,168
│    └─LeakyReLU: 2-10                   [1, 256, 56, 56]          --
│    └─Conv2d: 2-11                      [1, 256, 56, 56]          65,792
│    └─LeakyReLU: 2-12                   [1, 256, 56, 56]          --
│    └─Conv2d: 2-13                      [1, 512, 56, 56]          1,180,160
│    └─LeakyReLU: 2-14                   [1, 512, 56, 56]          --
│    └─MaxPool2d: 2-15                   [1, 512, 28, 28]          --
#▶ 크기: 56->28.
│    └─Conv2d: 2-16                      [1, 256, 28, 28]          131,328
│    └─... (LeakyReLU)
│    └─Conv2d: 2-18                      [1, 512, 28, 28]          1,180,160
#▶ 여기서부터 architecture_config의 첫 번째 반복 블록이 4번 실행되는 구간입니다.
#▶ (1x1 Conv -> 3x3 Conv) * 4
│    └─... (총 8개의 Conv 레이어가 반복됩니다)
│    └─Conv2d: 2-32                      [1, 512, 28, 28]          262,656
│    └─LeakyReLU: 2-33                   [1, 512, 28, 28]          --
│    └─Conv2d: 2-34                      [1, 1024, 28, 28]         4,719,616
│    └─LeakyReLU: 2-35                   [1, 1024, 28, 28]         --
│    └─MaxPool2d: 2-36                   [1, 1024, 14, 14]         --
#▶ 크기: 28->14.
│    └─Conv2d: 2-37                      [1, 512, 14, 14]          524,800
│    └─... (LeakyReLU)
│    └─Conv2d: 2-39                      [1, 1024, 14, 14]         4,719,616
#▶ 여기서부터 architecture_config의 두 번째 반복 블록이 2번 실행되는 구간입니다.
#▶ (1x1 Conv -> 3x3 Conv) * 2
│    └─... (총 4개의 Conv 레이어가 반복됩니다)
│    └─Conv2d: 2-45                      [1, 1024, 14, 14]         9,438,208
│    └─LeakyReLU: 2-46                   [1, 1024, 14, 14]         --
│    └─Conv2d: 2-47                      [1, 1024, 7, 7]           9,438,208
#▶ stride=2 Conv. 크기: 14->7.
│    └─... (LeakyReLU)
│    └─Conv2d: 2-49                      [1, 1024, 7, 7]           9,438,208
│    └─... (LeakyReLU)
│    └─Conv2d: 2-51                      [1, 1024, 7, 7]           9,438,208
#▶ 이 레이어가 컨볼루션 몸통의 마지막 레이어입니다.
#▶ 최종적으로 [1, 1024, 7, 7] 크기의 특징 맵을 생성하여 다음 단계로 넘깁니다.
│    └─LeakyReLU: 2-52                   [1, 1024, 7, 7]           --

# ------------------------------------------------------------------------------------------
# [2단계] 완전 연결 레이어 (Detection Head)
# ------------------------------------------------------------------------------------------
├─Sequential: 1-2                        [1, 1470]                 --
#▶ 이 Sequential 블록은 YOLOv1의 '머리' 부분으로, 추출된 특징을 해석하여 최종 예측 벡터를 만듭니다.
│    └─Flatten: 2-53                     [1, 50176]                --
#▶ 3D 특징 맵 [1, 1024, 7, 7]을 1D 벡터로 쭉 펼칩니다.
#▶ 크기: 1024 * 7 * 7 = 50,176
│    └─Linear: 2-54                      [1, 4096]                 205,524,992
#▶ 50176개 노드를 4096개 노드로 연결하는 첫 번째 FC 레이어.
#▶ 파라미터: 50176 * 4096 (가중치) + 4096 (편향) = 205,524,992
│    └─LeakyReLU: 2-55                   [1, 4096]                 --
│    └─Dropout: 2-56                     [1, 4096]                 --
#▶ 과적합 방지를 위한 Dropout. 파라미터 없음.
│    └─Linear: 2-57                      [1, 1470]                 6,022,590
#▶ 4096개 노드를 최종 출력 크기인 1470개 노드로 연결하는 마지막 FC 레이어.
#▶ 파라미터: 4096 * 1470 (가중치) + 1470 (편향) = 6,022,590
# ==========================================================================================
# [요약 정보]
# ==========================================================================================
Total params: 271,703,550
#▶ 모델의 총 파라미터 개수. YOLOv1은 FC 레이어의 파라미터가 매우 큰 것을 볼 수 있습니다. (약 2억 7천만 개)
Trainable params: 271,703,550
#▶ 이 중 학습 가능한 파라미터의 수입니다.
Non-trainable params: 0
#▶ 학습되지 않도록 고정된 파라미터의 수입니다.
Total mult-adds (Units.GIGABYTES): 20.30
#▶ 모델의 순전파에 필요한 총 연산량입니다. (약 20.3 Giga MACs)
# ==========================================================================================
# [메모리 사용량 예측]
# ==========================================================================================
Input size (MB): 2.41
#▶ 입력 텐서 [1, 3, 448, 448]가 차지하는 메모리 크기입니다.
Forward/backward pass size (MB): 110.43
#▶ 순전파/역전파 시 중간 결과물들을 저장하는 데 필요한 메모리입니다.
Params size (MB): 1086.81
#▶ 모델의 파라미터 자체를 저장하는 데 필요한 메모리입니다. (약 1GB)
Estimated Total Size (MB): 1199.65
#▶ 학습 시 필요한 총 GPU 메모리의 예측치입니다.
# ==========================================================================================

Layer (type:depth-idx)                   Output Shape              Param #
YOLOv1                                   [1, 1470]                 --
├─Sequential: 1-1                        [1, 1024, 7, 7]           --
│    └─Conv2d: 2-1                       [1, 64, 224, 224]         9,472
│    └─LeakyReLU: 2-2                    [1, 64, 224, 224]         --
│    └─MaxPool2d: 2-3                    [1, 64, 112, 112]         --
│    └─Conv2d: 2-4                       [1, 192, 112, 112]        110,784
│    └─LeakyReLU: 2-5                    [1, 192, 112, 112]        --
│    └─MaxPool2d: 2-6                    [1, 192, 56, 56]          --
│    └─Conv2d: 2-7                       [1, 128, 56, 56]          24,704
│    └─LeakyReLU: 2-8                    [1, 128, 56, 56]          --
│    └─Conv2d: 2-9                       [1, 256, 56, 56]          295,168
│    └─LeakyReLU: 2-10                   [1, 256, 56, 56]          --
│    └─Conv2d: 2-11                      [1, 256, 56, 56]          6